# Agent-Powered Gradio App with Human-in-the-Loop

*Optional notebook — capstone. Combines streaming, multi-turn state, and HITL into one app.*

A Gradio chat interface backed by an OpenAI Agents SDK agent. The agent can:
- Answer questions about course materials
- Perform calculations
- Send course announcements — but **only after human approval**

### The HITL checkpoint pattern

The agent has a `request_approval` tool. Calling it does not send anything — it signals
the UI to pause and show an Approve / Reject panel. The human's decision is fed back into
the conversation, and the agent continues from that point.

This is how production HITL works: structured checkpoints at decision boundaries,
not mid-run interruption.

In [1]:
import os
import json
import uuid
import numpy as np
import faiss
import gradio as gr
import logging
from openai import OpenAI
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, RunConfig, function_tool, RawResponsesStreamEvent
from agents.tracing import gen_trace_id
from pydantic import BaseModel

# Suppress verbose HTTP logs from the OpenAI client
logging.getLogger('httpx').setLevel(logging.WARNING)
logging.getLogger('openai').setLevel(logging.WARNING)
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()
print('Setup complete')


Setup complete


In [2]:
COURSE_TEXTS = [
    "Retrieval-Augmented Generation (RAG) combines a retrieval system with an LLM to ground answers in documents.",
    "FAISS is a library for efficient similarity search over dense vectors, built by Meta.",
    "Cosine similarity measures the angle between two vectors, returning a value between -1 and 1.",
    "An embedding is a dense numerical representation of text that captures semantic meaning.",
    "BM25 is a lexical search algorithm scoring by term frequency and inverse document frequency.",
    "The OpenAI Agents SDK uses Agent, Runner, function_tool, and handoffs as its core primitives.",
    "Agentic RAG lets an LLM iteratively retrieve and refine its answer rather than retrieving once.",
    "MCP (Model Context Protocol) is an open standard for connecting AI models to tools and data sources.",
    "Prompt chaining passes the output of one LLM call as the input to the next.",
    "The compound accuracy problem: a 10-step workflow at 85% accuracy per step succeeds only 20% of the time.",
]

def _embed(texts):
    r = client.embeddings.create(model="text-embedding-3-small", input=texts)
    return np.array([item.embedding for item in r.data], dtype=np.float32)

_vecs = _embed(COURSE_TEXTS)
_index = faiss.IndexFlatL2(_vecs.shape[1])
_index.add(_vecs)
print(f"Index ready: {_index.ntotal} vectors")

Index ready: 10 vectors


In [3]:
import math

@function_tool
def search_course_materials(query: str) -> str:
    """Search ANLP course materials for information relevant to the query. Returns 2 passages."""
    q = _embed([query]).reshape(1, -1)
    _, idx = _index.search(q, 2)
    return "\n\n".join(f"[{i+1}] {COURSE_TEXTS[j]}" for i, j in enumerate(idx[0]))

@function_tool
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression. Supports +, -, *, /, **, sqrt, log, sin, cos, pi, e."""
    safe_env = {k: getattr(math, k) for k in dir(math) if not k.startswith("_")}
    safe_env["__builtins__"] = {}
    try:
        return str(float(eval(expression, safe_env)))
    except Exception as e:
        return f"Error: {e}"

@function_tool
def request_approval(action: str, subject: str, body: str, recipients: str) -> str:
    """
    Request human approval before sending a course announcement.
    ALWAYS call this before send_course_announcement. Provide the full details.
    action: one-sentence description of what you want to do
    subject: email subject line
    body: full announcement text
    recipients: who will receive it (e.g. "all enrolled students")
    Returns 'AWAITING_APPROVAL' — the human will approve or reject via the UI.
    """
    return "AWAITING_APPROVAL"

@function_tool
def send_course_announcement(subject: str, body: str, recipients: str) -> str:
    """
    Send a course announcement. Only call this AFTER receiving human approval.
    If the human approved, proceed. If rejected, do not call this tool.
    """
    # In production this would call an email/LMS API
    return f"✓ Announcement sent to {recipients}.\nSubject: {subject}"

In [4]:
course_agent = Agent(
    name="Course Assistant",
    instructions="""You are a helpful assistant for an applied NLP course.

You can:
- Answer questions about course content using search_course_materials
- Perform calculations using calculate
- Send course announcements using send_course_announcement

IMPORTANT RULE FOR ANNOUNCEMENTS:
Before calling send_course_announcement, you MUST first call request_approval
with the full details of what you intend to send. Wait for the result.
- If the result is 'APPROVED': call send_course_announcement and confirm to the user.
- If the result is 'REJECTED': do NOT send. Tell the user it was rejected and ask what they would like instead.
""",
    tools=[search_course_materials, calculate, request_approval, send_course_announcement],
    model="gpt-4o-mini",
)
print(f"Agent ready: {course_agent.name}")

Agent ready: Course Assistant


## Gradio Event Handlers

Three async generator functions power the app:

- `on_send` — streams the agent response token by token; after streaming detects if
  `request_approval` was called and shows the HITL panel if so
- `on_approve` — appends "APPROVED" to history, streams agent's follow-up response
- `on_reject` — appends "REJECTED" to history, streams agent's follow-up response

All three use `to_input_list()` to preserve the full conversation across turns.

In [5]:
async def on_send(message, history, state):
    """Handle a user message: add to chat, stream agent response, detect approval request."""
    if not message.strip():
        yield history, gr.update(value=''), gr.update(interactive=True), gr.update(visible=False), '', state
        return

    # Generate a trace_id on the first message of this session.
    # The correlation_id was already assigned at session start (see gr.State init).
    if state['trace_id'] is None:
        state['trace_id'] = gen_trace_id()

    run_cfg = RunConfig(
        trace_id=state['trace_id'],
        workflow_name=f"gradio_session_{state['correlation_id']}",
        trace_metadata={'notebook': '04', 'correlation_id': state['correlation_id']},
    )

    # Show user message immediately
    history = history + [{'role': 'user', 'content': message}]
    yield history, gr.update(value=''), gr.update(interactive=False), gr.update(visible=False), '', state

    # Build input list: prior history + new user message
    input_list = state['input_list'] + [{'role': 'user', 'content': message}]

    # Stream agent response token by token
    partial = ''
    history = history + [{'role': 'assistant', 'content': ''}]

    streamed = Runner.run_streamed(course_agent, input_list, run_config=run_cfg)
    async for event in streamed.stream_events():
        if (isinstance(event, RawResponsesStreamEvent)
                and isinstance(event.data, ResponseTextDeltaEvent)):
            partial += event.data.delta
            history[-1]['content'] = partial
            yield history, gr.update(value=''), gr.update(interactive=False), gr.update(visible=False), '', state

    # Update conversation state
    state['input_list'] = streamed.to_input_list()

    # Detect if request_approval was called
    approval_args = None
    for item in streamed.new_items:
        if (item.type == 'tool_call_item'
                and hasattr(item.raw_item, 'name')
                and item.raw_item.name == 'request_approval'):
            approval_args = json.loads(item.raw_item.arguments)
            break

    if approval_args:
        state['pending_approval'] = approval_args
        detail_md = (
            f"**Action:** {approval_args.get('action', '')}\n\n"
            f"**Subject:** {approval_args.get('subject', '')}\n\n"
            f"**Recipients:** {approval_args.get('recipients', '')}\n\n"
            f"**Body:**\n{approval_args.get('body', '')}"
        )
        yield history, gr.update(value=''), gr.update(interactive=False), gr.update(visible=True), detail_md, state
    else:
        state['pending_approval'] = None
        yield history, gr.update(value=''), gr.update(interactive=True), gr.update(visible=False), '', state


In [6]:
async def on_decision(decision: str, history, state):
    """Stream agent response after human approves or rejects."""
    if decision == 'approve':
        human_msg = 'APPROVED — please proceed with sending the announcement.'
        label = '✓ Approved'
    else:
        human_msg = 'REJECTED — do not send. Ask me what I would like to do instead.'
        label = '✗ Rejected'

    history = history + [{'role': 'user', 'content': f'[{label}]'}]
    input_list = state['input_list'] + [{'role': 'user', 'content': human_msg}]

    run_cfg = RunConfig(
        trace_id=state['trace_id'],
        workflow_name=f"gradio_session_{state['correlation_id']}",
        trace_metadata={'notebook': '04', 'correlation_id': state['correlation_id']},
    )

    partial = ''
    history = history + [{'role': 'assistant', 'content': ''}]

    streamed = Runner.run_streamed(course_agent, input_list, run_config=run_cfg)
    async for event in streamed.stream_events():
        if (isinstance(event, RawResponsesStreamEvent)
                and isinstance(event.data, ResponseTextDeltaEvent)):
            partial += event.data.delta
            history[-1]['content'] = partial
            yield history, gr.update(visible=True), gr.update(interactive=False), state

    state['input_list'] = streamed.to_input_list()
    state['pending_approval'] = None
    yield history, gr.update(visible=False), gr.update(interactive=True), state

async def on_approve(history, state):
    async for update in on_decision('approve', history, state):
        yield update

async def on_reject(history, state):
    async for update in on_decision('reject', history, state):
        yield update


with gr.Blocks(title='Course Assistant — HITL Demo') as demo:
    state = gr.State({
        'input_list': [],
        'pending_approval': None,
        'trace_id': None,
        'correlation_id': None,
    })

    gr.Markdown("""
    # Course Assistant with Human-in-the-Loop
    Ask questions about the course, or ask the agent to send an announcement to students.
    When the agent proposes an announcement, you will be asked to approve or reject it before it is sent.
    """)

    chatbot = gr.Chatbot(
        height=450,
        buttons=['copy'],
        label='Conversation',
    )

    with gr.Row():
        msg_box = gr.Textbox(
            placeholder='Try: "Send a reminder to students about the Session 9 notebook deadline"',
            scale=5,
            show_label=False,
            lines=1,
        )
        send_btn = gr.Button('Send', variant='primary', scale=1)

    # HITL panel — hidden until agent calls request_approval
    with gr.Group(visible=False) as hitl_panel:
        gr.Markdown('## ⚠️ Agent Requesting Approval')
        approval_text = gr.Markdown('', label='Proposed Action')
        with gr.Row():
            approve_btn = gr.Button('✓ Approve', variant='primary', scale=1)
            reject_btn  = gr.Button('✗ Reject',  variant='stop',    scale=1)

    # Conversation correlation ID — shown to the user, searchable in OpenAI traces dashboard
    corr_id_box = gr.Textbox(
        label='Conversation ID (quote this when contacting support)',
        interactive=False,
    )

    # On page load: generate a unique correlation_id for this session,
    # store it in state, and display it in the textbox.
    def _init_session(s):
        s['correlation_id'] = uuid.uuid4().hex[:8].upper()
        return s, s['correlation_id']

    demo.load(_init_session, inputs=[state], outputs=[state, corr_id_box])

    # Wire up send — outputs: chatbot, msg_box (value), msg_box (interactive), hitl_panel, approval_text, state
    send_fn_outputs = [chatbot, msg_box, msg_box, hitl_panel, approval_text, state]
    send_btn.click(on_send, [msg_box, chatbot, state], send_fn_outputs)
    msg_box.submit(on_send, [msg_box, chatbot, state], send_fn_outputs)

    # Wire up approve / reject — outputs: chatbot, hitl_panel, msg_box, state
    decision_outputs = [chatbot, hitl_panel, msg_box, state]
    approve_btn.click(on_approve, [chatbot, state], decision_outputs)
    reject_btn.click(on_reject,   [chatbot, state], decision_outputs)

demo.launch()


In [7]:
def _make_state():
    """Called once per Gradio session — generates a unique correlation ID."""
    return {
        'input_list': [],
        'pending_approval': None,
        'trace_id': None,
        'correlation_id': uuid.uuid4().hex[:8].upper(),
    }

with gr.Blocks(title='Course Assistant — HITL Demo') as demo:
    state = gr.State(_make_state)

    gr.Markdown("""
    # Course Assistant with Human-in-the-Loop
    Ask questions about the course, or ask the agent to send an announcement to students.
    When the agent proposes an announcement, you will be asked to approve or reject it before it is sent.
    """)

    chatbot = gr.Chatbot(
        height=450,
        buttons=['copy'],
        label='Conversation',
    )

    with gr.Row():
        msg_box = gr.Textbox(
            placeholder='Try: "Send a reminder to students about the Session 9 notebook deadline"',
            scale=5,
            show_label=False,
            lines=1,
        )
        send_btn = gr.Button('Send', variant='primary', scale=1)

    # HITL panel — hidden until agent calls request_approval
    with gr.Group(visible=False) as hitl_panel:
        gr.Markdown('## ⚠️ Agent Requesting Approval')
        approval_text = gr.Markdown('', label='Proposed Action')
        with gr.Row():
            approve_btn = gr.Button('✓ Approve', variant='primary', scale=1)
            reject_btn  = gr.Button('✗ Reject',  variant='stop',    scale=1)

    # Conversation correlation ID — shown to user, searchable in OpenAI traces dashboard
    corr_id_box = gr.Textbox(
        label='Conversation ID (quote this when contacting support)',
        interactive=False,
        scale=1,
    )
    # Populate from state as soon as the page loads
    demo.load(lambda s: s['correlation_id'], inputs=[state], outputs=[corr_id_box])

    # Wire up send — outputs: chatbot, msg_box (value), msg_box (interactive), hitl_panel, approval_text, state
    send_fn_outputs = [chatbot, msg_box, msg_box, hitl_panel, approval_text, state]
    send_btn.click(on_send, [msg_box, chatbot, state], send_fn_outputs)
    msg_box.submit(on_send, [msg_box, chatbot, state], send_fn_outputs)

    # Wire up approve / reject — outputs: chatbot, hitl_panel, msg_box, state
    decision_outputs = [chatbot, hitl_panel, msg_box, state]
    approve_btn.click(on_approve, [chatbot, state], decision_outputs)
    reject_btn.click(on_reject,   [chatbot, state], decision_outputs)

demo.launch()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "c:\git\mdsiprojects\anlpaut2026\.venv\Lib\site-packages\gradio\queueing.py", line 766, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "c:\git\mdsiprojects\anlpaut2026\.venv\Lib\site-packages\gradio\route_utils.py", line 355, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<11 lines>...
    )
    ^
  File "c:\git\mdsiprojects\anlpaut2026\.venv\Lib\site-packages\gradio\blocks.py", line 2158, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<8 lines>...
    )
    ^
  File "c:\git\mdsiprojects\anlpaut2026\.venv\Lib\site-packages\gradio\blocks.py", line 1634, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        f

## Human-in-the-Loop as System Design

The HITL pattern here is not a fallback for when agents fail — it is the correct
architecture for agents operating at decision boundaries they should not cross alone.

| Decision type | Right approach |
|---|---|
| Answering a course question | Fully autonomous — low stakes, reversible |
| Performing a calculation | Fully autonomous — deterministic, verifiable |
| Sending a public announcement | HITL — high stakes, irreversible, visible to others |

The `request_approval` tool is the agent's way of saying:
*"I have everything I need to act, but this decision belongs to a human."*

This is the 2026 practitioner consensus: human oversight is architecture, not compromise.
Systems like Salesforce Agentforce and GitHub Copilot Workspace use exactly this pattern —
the agent proposes, the human confirms, the agent executes.